[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamtrask/abcGPT/blob/colab-slider/experiments/nano-3/slider_colab.ipynb)

# abcGPT nano-3 — 100-source slider (Colab)

A char-level GPT you can steer across up to **100 author sources** with **one slider per source**.
Pick a model (N=10 … 100), dial each source's weight, and blend any mixture. Pure one-hot is
cleanest (that's what they were trained on); rich blends are the interior and get rougher as N grows.

**Just run every cell top to bottom** (▶ each, or *Runtime → Run all*). Nothing to install or log in —
weights download from a public HuggingFace repo. For ~10× faster generation, optionally switch to a GPU:
*Runtime → Change runtime type → T4 GPU* (CPU works fine too).


In [ ]:
#@title 1. Setup — install deps + fetch model code (run once)
%pip install -q torch numpy huggingface_hub ipywidgets
import os, urllib.request
# the model definition is a single self-contained file on GitHub
if not os.path.exists("model.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/iamtrask/abcGPT/main/experiments/nano-3/model.py",
        "model.py")
    print("downloaded model.py")
# enable rich ipywidgets in Colab (no-op elsewhere)
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception:
    pass
print("setup done — run the next cell")


In [ ]:
#@title 2. Load weights + define the model loader (run once)
import pickle, tarfile, torch, numpy as np
import torch.nn.functional as F
from pathlib import Path
from huggingface_hub import hf_hub_download
from model import NanoGPT, NanoGPTConfig, LoRAAdditiveLinear, LoRAAdditiveEmbedding

HF_REPO = "iamtrask/abcGPT-nano-3"                       # public — no token needed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# char maps + cohort (source) names from the prebuilt 100-source bins
_tar = hf_hub_download(HF_REPO, "data_100src/bins.tar.gz", repo_type="model")
_md = Path("/tmp/abcgpt_100src"); _md.mkdir(parents=True, exist_ok=True)
if not (_md / "meta.pkl").exists():
    with tarfile.open(_tar) as t:
        mem = [x for x in t.getmembers()
               if x.name.endswith("meta.pkl") and not Path(x.name).name.startswith("._")][0]
        mem.name = Path(mem.name).name; t.extract(mem, _md)
META = pickle.load(open(_md / "meta.pkl", "rb"))
STOI, ITOS, ALLC = META["stoi"], META["itos"], META["cohort_names"]
VOCAB = META["vocab_size"]
def nice(c):
    return c.split("_", 2)[-1] if c.startswith("source_") else c

# per-α MERGE CACHE: the offload model recomputes each active cohort's delta PER
# TOKEN; for a dense 100-source blend that's slow. Patch the LoRA modules to merge
# the effective weight ONCE per distinct alpha (vectorized einsum) and reuse it.
def _merged_lin_W(m, alpha):
    if m.adaptive_capacity:
        caps = F.softmax(m.cohort_log_caps, 0) * m.n_cohorts
        base = torch.exp(m.base_log_cap) * m._W_shared(); w = alpha * caps
    else:
        base = m._W_shared(); w = alpha
    U = torch.stack(list(m.U)) if isinstance(m.U, torch.nn.ParameterList) else m.U
    V = torch.stack(list(m.V)) if isinstance(m.V, torch.nn.ParameterList) else m.V
    return base + torch.einsum('c,cor,cir->oi', w, U, V) * m.rslora_scale

def _cached_lin_forward(self, x, alpha):
    if alpha is None:
        return F.linear(x, self._W_shared(), self.bias)
    c = getattr(self, "_amerge", None)
    if c is None or not torch.equal(c[0], alpha):
        self._amerge = (alpha.detach().clone(), _merged_lin_W(self, alpha).detach())
    return F.linear(x, self._amerge[1], self.bias)
LoRAAdditiveLinear.forward = _cached_lin_forward

_orig_weff = LoRAAdditiveEmbedding._W_eff
def _cached_weff(self, alpha):
    c = getattr(self, "_amerge", None)
    if c is None or not torch.equal(c[0], alpha):
        self._amerge = (alpha.detach().clone(), _orig_weff(self, alpha).detach())
    return self._amerge[1]
LoRAAdditiveEmbedding._W_eff = _cached_weff

MODELS = {f"n{N}-offload-bR64-r64": N for N in (10,20,30,40,50,60,70,80,90,100)}
_cache = {}
def load_model(name):
    if name in _cache: return _cache[name]
    N = MODELS[name]
    p = hf_hub_download(HF_REPO, f"{name}/model.pt", repo_type="model")
    sd = torch.load(p, map_location="cpu", weights_only=True)
    block = sd["wpe.weight"].shape[0]
    cfg = NanoGPTConfig(vocab_size=VOCAB, block_size=block, n_layer=8, n_head=8, n_embd=512,
        dropout=0.0, n_cohorts=N, variant="lora", rank=64, base_rank=64,
        adaptive_capacity=True, bias_anchor=True, rslora=True,
        gate_attention=True, gate_embedding=True, offload_deltas=True)
    model = NanoGPT(cfg)
    miss, unexp = model.load_state_dict(sd, strict=False)
    if miss or unexp: print(f"  WARN {name}: missing {len(miss)} / unexpected {len(unexp)}")
    model.to(DEVICE).eval()
    _cache[name] = (model, cfg, ALLC[:N])
    print(f"loaded {name}  N={N}  ({model.num_params()/1e6:.0f}M, {block}-ctx)")
    return _cache[name]

@torch.no_grad()
def generate(model, cfg, alpha, prompt, max_new=180, temperature=0.8, top_k=40, seed=None):
    if seed is not None: torch.manual_seed(seed)
    ids = [STOI.get(ch, STOI.get(" ", 0)) for ch in prompt]
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    a = torch.tensor(alpha, dtype=torch.float32, device=DEVICE)
    out = list(ids)
    for _ in range(max_new):
        xc = x[:, -cfg.block_size:]
        logits, _ = model(xc, a, None)
        logits = logits[:, -1, :].float() / max(1e-6, temperature)
        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -1e9
        nx = torch.multinomial(F.softmax(logits, dim=-1), 1)
        x = torch.cat([x, nx], dim=1); out.append(int(nx.item()))
    return "".join(ITOS.get(i, "?") for i in out)

print("ready — run the slider cell")


In [ ]:
#@title 3. The slider — one weight per source. Run, then mix.
import ipywidgets as W
from IPython.display import display

model_dd = W.Dropdown(options=list(MODELS), value="n100-offload-bR64-r64", description="model")
prompt   = W.Textarea(value="the ", description="prompt", layout=W.Layout(width="640px", height="50px"))
temp     = W.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.05, description="temp")
length   = W.IntSlider(value=180, min=40, max=400, step=20, description="tokens")
topk     = W.IntSlider(value=40, min=0, max=VOCAB, step=1, description="top_k")
normalize = W.Checkbox(value=True, description="normalize α → sum 1")
clear_b = W.Button(description="clear"); unif_b = W.Button(description="uniform")
rand_b  = W.Button(description="random"); gen_b = W.Button(description="Generate", button_style="primary")
out = W.Output()
src_sliders = []
panel  = W.VBox([])
scroll = W.Box([panel], layout=W.Layout(overflow="auto", height="360px",
                                        border="1px solid #888", padding="4px"))

def rebuild(*_):
    global src_sliders
    cohorts = ALLC[:MODELS[model_dd.value]]
    src_sliders = [W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description=nice(c),
                       readout_format=".2f", continuous_update=False,
                       style={"description_width": "200px"}, layout=W.Layout(width="440px"))
                   for c in cohorts]
    if src_sliders: src_sliders[0].value = 1.0
    panel.children = src_sliders
model_dd.observe(rebuild, names="value")

def _set_all(vals):
    for s, v in zip(src_sliders, vals): s.value = round(float(v), 2)
clear_b.on_click(lambda b: _set_all([0.0] * len(src_sliders)))
unif_b.on_click(lambda b: _set_all([1.0 / len(src_sliders)] * len(src_sliders)))
def _rand(b):
    import random
    v = [random.random() ** 3 for _ in src_sliders]; s = sum(v) or 1.0
    _set_all([x / s for x in v])
rand_b.on_click(_rand)

def on_gen(b):
    with out:
        out.clear_output()
        m, cfg, cohorts = load_model(model_dd.value)
        raw = np.array([s.value for s in src_sliders], dtype=np.float32)
        if raw.sum() == 0:
            print("all sliders at 0 — raise at least one source."); return
        alpha = raw / raw.sum() if normalize.value else raw
        top = sorted([(nice(cohorts[i]), float(a)) for i, a in enumerate(alpha) if a > 0],
                     key=lambda kv: -kv[1])[:8]
        print(f"model: {model_dd.value} | N={len(cohorts)} | {int((alpha>0).sum())} active | "
              + ", ".join(f"{k}={v:.2f}" for k, v in top))
        print("(merging deltas at this mix — first gen of a new mix takes a moment) ...")
        print("-" * 72)
        print(generate(m, cfg, alpha.tolist(), prompt.value, max_new=length.value,
                       temperature=temp.value, top_k=topk.value))
gen_b.on_click(on_gen)
rebuild()
display(W.VBox([model_dd,
               W.HTML("<b>per-source α sliders</b> — one per source for the chosen model (scroll):"),
               scroll, W.HBox([clear_b, unif_b, rand_b, normalize]),
               prompt, temp, length, topk, gen_b, out]))
